In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load the image
image = cv2.imread('bills.jpg')
if image is None:
    raise FileNotFoundError("Image not found.")

H, W = image.shape[:2]   # work at the photo's original resolution (no resizing)

# Convert to RGB (for display) and HSV (for color detection)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
plt.figure(figsize=(10, 8))
plt.imshow(image_rgb)
plt.title(f"Original image ({W}x{H})")
plt.axis("off")
plt.show()


In [ ]:
# HSV window(s) for each denomination. A denomination can list several windows,
# e.g. red sits at both ends of the hue circle.
bill_colors = {
    'Orange':   {'ranges': [((10,   100, 100), (17,  255, 255))],
                    'box_color': (255, 140, 0),   'value': 20},
    'Red':      {'ranges': [((175, 100, 100), (180, 255, 255))],
                    'box_color': (255, 0, 0),     'value': 50},
    'Violet':   {'ranges': [((140, 40,  50), (162, 255, 220))],
                    'box_color': (148, 0, 211),   'value': 100},
    'Green':    {'ranges': [((40,  35,  40), (60,  255, 255))],
                    'box_color': (0, 150, 0),     'value': 200},
    'Yellow':   {'ranges': [((18,  90, 90), (27,  255, 255))],
                    'box_color': (255, 215, 0),   'value': 500},
    'Blue':     {'ranges': [((92,  60,  50), (114, 255, 220))],
                    'box_color': (0, 0, 255),     'value': 1000},
}

def build_mask(hsv_img, ranges):
    """Union of all HSV windows for one denomination, as a 0/255 uint8 mask."""
    parts = [cv2.inRange(hsv_img, np.array(lo), np.array(hi)) for lo, hi in ranges]
    return np.bitwise_or.reduce(parts)

# Preview the raw colour mask for every denomination
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, info) in zip(axs.ravel(), bill_colors.items()):
    ax.imshow(build_mask(hsv, info['ranges']), cmap='gray')
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Detection settings, all relative to image size ---------------------------
def odd(n):
    n = int(n)
    return n if n % 2 else n + 1

def ellipse(size):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))

short_side = min(H, W)
speck_kernel = ellipse(3)                              # removes isolated noise pixels
bridge_kernel = ellipse(odd(max(9, short_side * 0.05)))  # joins printed areas split by white gaps
grow_kernel = ellipse(odd(max(5, short_side * 0.03)))    # expands the coloured patch toward the bill edge
MIN_AREA = 0.01 * H * W                                # ignore blobs under 1% of the image

def clean_mask(mask):
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, speck_kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, bridge_kernel, iterations=2)
    return cv2.dilate(mask, grow_kernel)

def find_bills(name, info):
    """Return one detection per large connected colour blob for this denomination."""
    raw = build_mask(hsv, info['ranges'])
    count, labels, stats, _ = cv2.connectedComponentsWithStats(clean_mask(raw), connectivity=8)
    # Number of genuinely coloured pixels in each blob (measured before the morphology grew it)
    colored_px = np.bincount(labels[raw > 0], minlength=count)
    found = []
    for i in range(1, count):                          # label 0 is the background
        x, y, bw, bh, area = stats[i]
        if area >= MIN_AREA:
            found.append({'name': name, 'value': info['value'],
                          'box': (int(x), int(y), int(bw), int(bh)),
                          'pixels': int(colored_px[i])})
    return found

def overlap_ratio(a, b):
    """Intersection area divided by the SMALLER box's area (1.0 = one box fully inside the other)."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    iw = min(ax + aw, bx + bw) - max(ax, bx)
    ih = min(ay + ah, by + bh) - max(ay, by)
    if iw <= 0 or ih <= 0:
        return 0.0
    return iw * ih / min(aw * ah, bw * bh)

def union_box(a, b):
    x0, y0 = min(a[0], b[0]), min(a[1], b[1])
    x1, y1 = max(a[0] + a[2], b[0] + b[2]), max(a[1] + a[3], b[1] + b[3])
    return (x0, y0, x1 - x0, y1 - y0)

OVERLAP_THRESHOLD = 0.5   # boxes sharing more than this fraction of the smaller box = same bill

def resolve_overlaps(dets):
    """A bill's secondary colours also form blobs inside it. When detections overlap,
    the colour with the most pixels owns the bill; the others are merged into its box."""
    kept = []
    for d in sorted(dets, key=lambda d: d['pixels'], reverse=True):
        for k in kept:
            if overlap_ratio(d['box'], k['box']) >= OVERLAP_THRESHOLD:
                k['box'] = union_box(k['box'], d['box'])
                break
        else:
            kept.append(dict(d))
    return kept

all_blobs = [d for name, info in bill_colors.items() for d in find_bills(name, info)]
detections = resolve_overlaps(all_blobs)
detections.sort(key=lambda d: d['box'][0])             # left-to-right

print(f"Found {len(all_blobs)} colour blob(s) -> {len(detections)} bill(s) after merging overlaps.")


In [ ]:
# Draw a labelled box on every detection
output_img = image_rgb.copy()
thickness = max(2, short_side // 200)
font_scale = max(0.5, short_side / 1000)

for d in detections:
    x, y, bw, bh = d['box']
    color = bill_colors[d['name']]['box_color']
    text = f"PHP {d['value']}"
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)

    cv2.rectangle(output_img, (x, y), (x + bw, y + bh), color, thickness)
    cv2.rectangle(output_img, (x, y), (x + tw + 12, y + th + base + 10), color, -1)   # tag inside the box
    cv2.putText(output_img, text, (x + 6, y + th + 5),
                cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness)

plt.figure(figsize=(12, 9))
plt.imshow(output_img)
plt.title("Detected bills")
plt.axis("off")
plt.show()

# Summary per denomination and overall total
print(f"{'Bill':<9}{'Colour':<10}{'Count':>6}{'Subtotal':>12}")
total_value = 0
for name, info in sorted(bill_colors.items(), key=lambda kv: kv[1]['value']):
    n = sum(d['name'] == name for d in detections)
    total_value += n * info['value']
    print(f"PHP {info['value']:<5}{name:<10}{n:>6}{n * info['value']:>12,}")

print(f"\nTotal count: {len(detections)}")
print(f"Total value: PHP {total_value:,.2f}")